In [123]:
import os
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv

In [71]:
load_dotenv()

SEOUL_API_KEY = os.getenv("SEOUL_API_KEY")

In [72]:
SEOUL_API_KEY is not None

True

# 데이터 불러오기

## 테스트

In [73]:
url = (
    f"http://openapi.seoul.go.kr:8088/"
    f"{SEOUL_API_KEY}/json/culturalEventInfo/1/5/"
)

response = requests.get(url)

response.status_code

200

In [74]:
data = response.json()

data.keys()

dict_keys(['culturalEventInfo'])

In [75]:
data["culturalEventInfo"].keys()

dict_keys(['list_total_count', 'RESULT', 'row'])

In [76]:
rows = data["culturalEventInfo"]["row"]

In [77]:
df = pd.DataFrame(rows)

df.head()

,CODENAME,GUNAME,TITLE,DATE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,INQUIRY,PLAYER,...,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,HMPG_ADDR,PRO_TIME
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",2026-12-24~2026-12-24,강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-23,시민,2026-12-24 00:00:00.0,2026-12-24 00:00:00.0,기타,127.157342546961,37.5512204558342,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",2026-12-22~2026-12-22,영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-16,시민,2026-12-22 00:00:00.0,2026-12-22 00:00:00.0,기타,126.900109255921,37.5260087284496,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],2026-11-29~2026-11-29,마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),"02-3274-8600 [문의1번] 평일 9:00 ~ 18:00 (토,일 공휴일 휴무)",,...,2026-08-04,기관,2026-11-29 00:00:00.0,2026-11-29 00:00:00.0,기타,126.9455874749264,37.54987259578174,유료,https://culture.seoul.go.kr/culture/culture/cu...,(일) 16:00
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),2026-11-27~2026-11-29,동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,,031-921-6325,,...,2026-07-21,시민,2026-11-27 00:00:00.0,2026-11-29 00:00:00.0,기타,127.00977973484339,37.56735731522952,무료,https://culture.seoul.go.kr/culture/culture/cu...,10:00 ~ 19:00
4,콘서트,강북구,[꿈의숲아트센터] 꿈의숲 마티네 콘서트 [벨에포크 아트&뮤직] 시리즈3,2026-10-28~2026-10-28,북서울꿈의숲 상상톡톡미술관,세종문화회관,8세 이상 관람 가능,"전석 15,000원",02-399-1000,,...,2026-06-30,기관,2026-10-28 00:00:00.0,2026-10-28 00:00:00.0,기타,127.044324732036,37.6202544613023,유료,https://culture.seoul.go.kr/culture/culture/cu...,수요일 11:00


In [78]:
df.columns

Index(['CODENAME', 'GUNAME', 'TITLE', 'DATE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'INQUIRY', 'PLAYER', 'PROGRAM', 'ETC_DESC', 'ORG_LINK',
       'MAIN_IMG', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'HMPG_ADDR', 'PRO_TIME'],
      dtype='str')

## 분석 데이터 불러오기

In [79]:
total_count = data["culturalEventInfo"]["list_total_count"]
all_rows = []

for start in range(1, total_count + 1, 1000):
    end = min(start + 999, total_count)

    url = (
        f"http://openapi.seoul.go.kr:8088/"
        f"{SEOUL_API_KEY}/json/culturalEventInfo/"
        f"{start}/{end}/"
    )

    response = requests.get(url)
    result = response.json()

    rows = result["culturalEventInfo"]["row"]
    all_rows.extend(rows)

In [80]:
df = pd.DataFrame(all_rows)
df.shape

(19502, 24)

In [81]:
print("API 전체 데이터 :", total_count)
print("실제 수집 데이터 :", len(df))
print("데이터 크기 :", df.shape)
print("중복 행 :", df.duplicated().sum())

API 전체 데이터 : 19502
실제 수집 데이터 : 19502
데이터 크기 : (19502, 24)
중복 행 : 0


# 데이터 전처리

## 불필요 컬럼 제거

In [82]:
df.head()

,CODENAME,GUNAME,TITLE,DATE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,INQUIRY,PLAYER,...,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,HMPG_ADDR,PRO_TIME
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",2026-12-24~2026-12-24,강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-23,시민,2026-12-24 00:00:00.0,2026-12-24 00:00:00.0,기타,127.157342546961,37.5512204558342,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",2026-12-22~2026-12-22,영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-16,시민,2026-12-22 00:00:00.0,2026-12-22 00:00:00.0,기타,126.900109255921,37.5260087284496,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],2026-11-29~2026-11-29,마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),"02-3274-8600 [문의1번] 평일 9:00 ~ 18:00 (토,일 공휴일 휴무)",,...,2026-08-04,기관,2026-11-29 00:00:00.0,2026-11-29 00:00:00.0,기타,126.9455874749264,37.54987259578174,유료,https://culture.seoul.go.kr/culture/culture/cu...,(일) 16:00
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),2026-11-27~2026-11-29,동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,,031-921-6325,,...,2026-07-21,시민,2026-11-27 00:00:00.0,2026-11-29 00:00:00.0,기타,127.00977973484339,37.56735731522952,무료,https://culture.seoul.go.kr/culture/culture/cu...,10:00 ~ 19:00
4,콘서트,강북구,[꿈의숲아트센터] 꿈의숲 마티네 콘서트 [벨에포크 아트&뮤직] 시리즈3,2026-10-28~2026-10-28,북서울꿈의숲 상상톡톡미술관,세종문화회관,8세 이상 관람 가능,"전석 15,000원",02-399-1000,,...,2026-06-30,기관,2026-10-28 00:00:00.0,2026-10-28 00:00:00.0,기타,127.044324732036,37.6202544613023,유료,https://culture.seoul.go.kr/culture/culture/cu...,수요일 11:00


In [83]:
df.columns

Index(['CODENAME', 'GUNAME', 'TITLE', 'DATE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'INQUIRY', 'PLAYER', 'PROGRAM', 'ETC_DESC', 'ORG_LINK',
       'MAIN_IMG', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'HMPG_ADDR', 'PRO_TIME'],
      dtype='str')

In [84]:
df = df[['CODENAME', 'GUNAME', 'TITLE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'PRO_TIME', 'ORG_LINK', 'HMPG_ADDR']]

## 데이터 타입 변경

In [85]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19502 entries, 0 to 19501
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   CODENAME   19502 non-null  str  
 1   GUNAME     19502 non-null  str  
 2   TITLE      19502 non-null  str  
 3   PLACE      19502 non-null  str  
 4   ORG_NAME   19502 non-null  str  
 5   USE_TRGT   19502 non-null  str  
 6   USE_FEE    19502 non-null  str  
 7   RGSTDATE   19502 non-null  str  
 8   TICKET     19502 non-null  str  
 9   STRTDATE   19502 non-null  str  
 10  END_DATE   19502 non-null  str  
 11  THEMECODE  19502 non-null  str  
 12  LOT        19502 non-null  str  
 13  LAT        19502 non-null  str  
 14  IS_FREE    19502 non-null  str  
 15  PRO_TIME   19502 non-null  str  
 16  ORG_LINK   19502 non-null  str  
 17  HMPG_ADDR  19502 non-null  str  
dtypes: str(18)
memory usage: 2.7 MB


In [86]:
df['STRTDATE'] = pd.to_datetime(df['STRTDATE'])
df['END_DATE'] = pd.to_datetime(df['END_DATE'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19502 entries, 0 to 19501
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   CODENAME   19502 non-null  str           
 1   GUNAME     19502 non-null  str           
 2   TITLE      19502 non-null  str           
 3   PLACE      19502 non-null  str           
 4   ORG_NAME   19502 non-null  str           
 5   USE_TRGT   19502 non-null  str           
 6   USE_FEE    19502 non-null  str           
 7   RGSTDATE   19502 non-null  str           
 8   TICKET     19502 non-null  str           
 9   STRTDATE   19502 non-null  datetime64[us]
 10  END_DATE   19502 non-null  datetime64[us]
 11  THEMECODE  19502 non-null  str           
 12  LOT        19502 non-null  str           
 13  LAT        19502 non-null  str           
 14  IS_FREE    19502 non-null  str           
 15  PRO_TIME   19502 non-null  str           
 16  ORG_LINK   19502 non-null  str           
 17  HMPG

In [87]:
df.head()

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",2026-07-23,시민,2026-12-24,2026-12-24,기타,127.157342546961,37.5512204558342,유료,19:30,https://tickets.interpark.com/goods/26010350,https://culture.seoul.go.kr/culture/culture/cu...
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",2026-07-16,시민,2026-12-22,2026-12-22,기타,126.900109255921,37.5260087284496,유료,19:30,https://tickets.interpark.com/goods/26010060,https://culture.seoul.go.kr/culture/culture/cu...
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),2026-08-04,기관,2026-11-29,2026-11-29,기타,126.9455874749264,37.54987259578174,유료,(일) 16:00,https://www.mfac.or.kr/performance/whole_view....,https://culture.seoul.go.kr/culture/culture/cu...
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,,2026-07-21,시민,2026-11-27,2026-11-29,기타,127.00977973484339,37.56735731522952,무료,10:00 ~ 19:00,https://finecharacter.kr/,https://culture.seoul.go.kr/culture/culture/cu...
4,콘서트,강북구,[꿈의숲아트센터] 꿈의숲 마티네 콘서트 [벨에포크 아트&뮤직] 시리즈3,북서울꿈의숲 상상톡톡미술관,세종문화회관,8세 이상 관람 가능,"전석 15,000원",2026-06-30,기관,2026-10-28,2026-10-28,기타,127.044324732036,37.6202544613023,유료,수요일 11:00,https://www.sejongpac.or.kr/dfac/dfacPerforman...,https://culture.seoul.go.kr/culture/culture/cu...


## 결측값 체크

In [88]:
df.isna().sum()

CODENAME     0
GUNAME       0
TITLE        0
PLACE        0
ORG_NAME     0
USE_TRGT     0
USE_FEE      0
RGSTDATE     0
TICKET       0
STRTDATE     0
END_DATE     0
THEMECODE    0
LOT          0
LAT          0
IS_FREE      0
PRO_TIME     0
ORG_LINK     0
HMPG_ADDR    0
dtype: int64

In [89]:
print("빈 문자열 개수")
print((df == "").sum())

빈 문자열 개수
CODENAME         0
GUNAME         124
TITLE            0
PLACE            0
ORG_NAME         0
USE_TRGT         0
USE_FEE      10501
RGSTDATE         0
TICKET           0
STRTDATE         0
END_DATE         0
THEMECODE     3329
LOT            521
LAT            521
IS_FREE          8
PRO_TIME         0
ORG_LINK       477
HMPG_ADDR        0
dtype: int64


In [90]:
len(df['USE_TRGT'].unique())

5008

In [91]:
df["USE_TRGT"].value_counts().head(30)

USE_TRGT
누구나                           4222
시민 누구나                         734
전체관람가                          677
성인                             482
홈페이지 참고                        469
초등학생 이상                        318
만 7세 이상                        302
8세 이상                          284
어린이                            166
초등학생 이상 관람가                    164
전체 관람가                         153
관심있는 누구나                       150
전체                             148
미취학아동 입장불가                     125
전 연령                           125
8세 이상 관람가                      107
36개월 이상                        104
프로그램별 상이                       104
7세 이상                           95
7세 이상 관람 가능 (2018년 이전 출생자)      89
모든 시민                           87
전연령                             82
5세 이상 어린이                       67
누구나                             66
7세 이상 관람 가능 (2019년 이전 출생자)      63
서울도서관 회원                        62
만 7세 이상                         59
일반시민                            58
초등학생       

In [92]:
df[df["USE_TRGT"] == "홈페이지 참고"]['HMPG_ADDR']

419      https://culture.seoul.go.kr/culture/culture/cu...
1101     https://culture.seoul.go.kr/culture/culture/cu...
2306     https://culture.seoul.go.kr/culture/culture/cu...
2515     https://culture.seoul.go.kr/culture/culture/cu...
3269     https://culture.seoul.go.kr/culture/culture/cu...
                               ...                        
18389    https://culture.seoul.go.kr/culture/culture/cu...
18400    https://culture.seoul.go.kr/culture/culture/cu...
18454    https://culture.seoul.go.kr/culture/culture/cu...
18573    https://culture.seoul.go.kr/culture/culture/cu...
18596    https://culture.seoul.go.kr/culture/culture/cu...
Name: HMPG_ADDR, Length: 469, dtype: str

행사 대상이 불명확한 데이터가 존재 함.


In [93]:
df["event_year"] = df["STRTDATE"].dt.year

df["event_year"].value_counts().sort_index()

event_year
2021     926
2022    3075
2023    3567
2024    5537
2025    3918
2026    2479
Name: count, dtype: int64

### 행사 시작일이 24년부터 현재까지의 데이터와 향후 예정인 데이터 대상으로 분석

In [94]:
df["STRTDATE"] = pd.to_datetime(
    df["STRTDATE"],
    errors="coerce"
)

start_date = pd.Timestamp("2024-01-01")

df_recent = df[
    df["STRTDATE"] >= start_date
].copy()

In [95]:
print("전체 행사 :", len(df))
print("최근 2년 행사 :", len(df_recent))

print(
    df_recent["STRTDATE"].min(),
    df_recent["STRTDATE"].max())

전체 행사 : 19502
최근 2년 행사 : 11934
2024-01-01 00:00:00 2026-12-24 00:00:00


# raw 데이터 내보내기

In [ ]:
df.to_csv(
    "../data/cultural_events_raw.csv",
    index=False
)

In [ ]:
df_recent.to_csv(
    "../data/cultural_events_2024_present.csv",
    index=False
)

In [100]:
df_c = pd.read_csv("C:\\workspace\\sprint_mission17\\data\\cultural_events_2024_present.csv")
df_c

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",2026-07-23,시민,2026-12-24,2026-12-24,기타,127.157342546961,37.5512204558342,유료,19:30,https://tickets.interpark.com/goods/26010350,https://culture.seoul.go.kr/culture/culture/cu...,2026
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",2026-07-16,시민,2026-12-22,2026-12-22,기타,126.900109255921,37.5260087284496,유료,19:30,https://tickets.interpark.com/goods/26010060,https://culture.seoul.go.kr/culture/culture/cu...,2026
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),2026-08-04,기관,2026-11-29,2026-11-29,기타,126.9455874749264,37.54987259578174,유료,(일) 16:00,https://www.mfac.or.kr/performance/whole_view....,https://culture.seoul.go.kr/culture/culture/cu...,2026
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,NaN,2026-07-21,시민,2026-11-27,2026-11-29,기타,127.00977973484339,37.56735731522952,무료,10:00 ~ 19:00,https://finecharacter.kr/,https://culture.seoul.go.kr/culture/culture/cu...,2026
4,콘서트,강북구,[꿈의숲아트센터] 꿈의숲 마티네 콘서트 [벨에포크 아트&뮤직] 시리즈3,북서울꿈의숲 상상톡톡미술관,세종문화회관,8세 이상 관람 가능,"전석 15,000원",2026-06-30,기관,2026-10-28,2026-10-28,기타,127.044324732036,37.6202544613023,유료,수요일 11:00,https://www.sejongpac.or.kr/dfac/dfacPerforman...,https://culture.seoul.go.kr/culture/culture/cu...,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11929,교육/체험,중구,"[서울시립미술관] SeMA L [모으다-잇다, 마음구슬], [꼬리에 꼬리를 무는 미...",서울시립미술관 서소문본관 3층 프로젝트갤러리,서울시립미술관,누구나,NaN,2024-04-30,기관,2024-01-01,2024-12-31,기타,126.973699316136,37.5641060692766,무료,평일(화-금)오전 10시-오후 8시 토 · 일 · 공휴일 오전 10시-오후 7시 (...,https://sema.seoul.go.kr/kr/whatson/education/...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11930,전시/미술,종로구,동대문 역사관,"2호선 동대문역사문화공원 1번, 2번 출구 / 4호선 동대문역 7번 출구",동대문디자인플라자,누구나,NaN,2024-01-02,기관,2024-01-01,2024-12-31,기타,127.01153416188667,37.56710262553757,무료,10:00 ~ 18:00 (※ 12~13시 휴관),https://ddp.or.kr/index.html?menuno=239&siteno...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11931,전시/미술,중구,[DDP] 테테루뮤지엄 홀로그램 전시관,"DDP디자인랩 1층 C5호, C6호",동대문디자인플라자,전체관람가,"입장권 : 성인 8,000원, 유아동 및 초중고학생 5,000원",2024-10-16,기관,2024-01-01,2024-12-31,기타,127.00977973484339,37.56735731522952,유료,화-금(11:00-17:00) / 토-일(11:00-17:30) / 월요일휴관,https://ddp.or.kr/index.html?menuno=239&siteno...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11932,교육/체험,송파구,2024 제1분기 송파문화원 수강생 모집,송파문화원,송파문화원,시민 누구나,프로그램별 상이,2023-12-29,기관,2024-01-01,2024-03-23,기타,127.075936810272,37.5106823860183,유료,프로그램별 상이,https://www.spcc.or.kr/spcccontents.asp?cc=050...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [102]:
df_c.isna().sum()

CODENAME         0
GUNAME          69
TITLE            0
PLACE            0
ORG_NAME         0
USE_TRGT         0
USE_FEE       6632
RGSTDATE         0
TICKET           0
STRTDATE         0
END_DATE         0
THEMECODE       65
LOT              1
LAT              1
IS_FREE          0
PRO_TIME         0
ORG_LINK         0
HMPG_ADDR        0
event_year       0
dtype: int64

# 2차 전처리

- 1차 전처리 후 당시 결측이 확인되지 않았으나, 내보낸 파일을 열었을 때 결측이 확인 됨에 따라 해당 결측처리를 진행함.
- 분석 및 대시보드 제작에 필요한 데이터 위주로 처리 (GUNAME, LOT, LAT)
- 좌표 데이터에 이상치 "-", "~" 데이터 확인 처리 필요.

## 결측 데이터 확인

In [103]:
df_c[df_c['GUNAME'].isna()]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
197,전시/미술,NaN,"[서울시립미술관] 아마도, 모두 우리 [Perhaps, All of Us] 캐나다 전시",주캐나다한국문화원 1층 전시실,서울시립미술관,누구나,NaN,2026-08-10,기관,2026-08-25,2026-10-14,기타,75.6917°W,45.4215°N,무료,월 9:00~17:00 화-금 9:00~20:00 토 10:00~18:00 ※ 매주...,https://sema.seoul.go.kr/kr/whatson/exhibition...,https://culture.seoul.go.kr/culture/culture/cu...,2026
963,전시/미술,NaN,"[서울시립미술관] 2026 투어링 케이-아츠 《아마도, 모두 우리》","주워싱턴한국문화원 (2370 Massachusetts Ave., N.W. Washi...",서울시립미술관,누구나,NaN,2026-06-10,기관,2026-06-17,2026-08-11,기타,126.973699316136,37.5641060692766,무료,10:00 ~ 17:00 (12:00 ~ 13:00 관람 제한) 주말 및 미국 법정...,https://sema.seoul.go.kr/kr/whatson/exhibition...,https://culture.seoul.go.kr/culture/culture/cu...,2026
1840,축제-자연/경관,NaN,[서울대공원] 2026년 봄꽃축제,서울대공원,서울대공원,누구나,입장료 별도,2026-03-24,기관,2026-04-04,2026-04-12,기타,127.014098361931,37.4364305503019,유료,9:00 ~ 18:00,https://grandpark.seoul.go.kr/munhwa/munhwaVie...,https://culture.seoul.go.kr/culture/culture/cu...,2026
2453,교육/체험,NaN,[서울대공원] 겨울을 녹이는 남미관 이야기,서울동물원 남미관,서울대공원,"어린이, 청소년, 성인 등 누구나 참여가능",※ 입장료 별도,2026-01-28,기관,2026-01-05,2026-02-07,기타,127.014098361931,37.4364305503019,유료,1차 10:40~11:10 / 2차 13:20~13:50 / 3차 14:00~14:...,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2026
3236,축제-문화/예술,NaN,[서울대공원] 2025 가을단풍축제 [가을빛 대공원],서울대공원 만남의 광장 일대,서울대공원,누구나,입장료 별도,2025-10-21,기관,2025-10-25,2025-11-02,기타,127.014098361931,37.4364305503019,유료,12:00 ~ 17:00,https://grandpark.seoul.go.kr/munhwa/munhwaVie...,https://culture.seoul.go.kr/culture/culture/cu...,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11443,교육/체험,NaN,[서울대공원] 물범 친구들을 소개합니다!,서울대공원 해양관,서울대공원,초등3~6학년,공원입장료,2024-02-27,기관,2024-03-09,2024-03-30,어린이/청소년 문화행사,127.014098361931,37.4364305503019,유료,13:20~15:00,https://grandpark.seoul.go.kr/edu/program/view...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11444,교육/체험,NaN,[서울대공원] 물범 친구들을 소개합니다,서울동물원 해양관,서울대공원,어린이,동물원 입장료 별도,2024-02-08,기관,2024-03-09,2024-03-30,어린이/청소년 문화행사,127.014098361931,37.4364305503019,무료,매주 토 13:20~15:00,https://grandpark.seoul.go.kr/conts/contsView/...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11662,교육/체험,NaN,[서울대공원] 2024 갑진년(甲辰年) 푸른 용과 멸종위기 동물 구해용!,서울대공원 동물원 제1아프리카관,서울대공원,어린이 등 관람객,NaN,2024-01-31,기관,2024-02-11,2024-02-18,어린이/청소년 문화행사,127.014098361931,37.4364305503019,무료,1차 11:00 2차 13:20 3차 14:00 4차 14:40,https://yeyak.seoul.go.kr/web/reservation/sele...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11725,클래식,NaN,서울시립교향악단 특별 음악회 얍 판 츠베덴의 바그너 [발퀴레],세종예술의전당,서울시립교향악단,8세이상 관람가능,"VIP석 100,000원, R석 80,000원, S석 60,000원",2024-01-09,시민,2024-02-02,2024-02-02,기타,127.267380381461,36.4866171885619,유료,19:30,https://www.seoulphil.or.kr/perf/view?perfNo=5...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [104]:
missing_gu = df_c[
    df_c["GUNAME"].isna()
]

missing_gu[
    ["TITLE", "PLACE", "GUNAME", "LAT", "LOT"]
].head(20)

,TITLE,PLACE,GUNAME,LAT,LOT
197,"[서울시립미술관] 아마도, 모두 우리 [Perhaps, All of Us] 캐나다 전시",주캐나다한국문화원 1층 전시실,NaN,45.4215°N,75.6917°W
963,"[서울시립미술관] 2026 투어링 케이-아츠 《아마도, 모두 우리》","주워싱턴한국문화원 (2370 Massachusetts Ave., N.W. Washi...",NaN,37.5641060692766,126.973699316136
1840,[서울대공원] 2026년 봄꽃축제,서울대공원,NaN,37.4364305503019,127.014098361931
2453,[서울대공원] 겨울을 녹이는 남미관 이야기,서울동물원 남미관,NaN,37.4364305503019,127.014098361931
3236,[서울대공원] 2025 가을단풍축제 [가을빛 대공원],서울대공원 만남의 광장 일대,NaN,37.4364305503019,127.014098361931
4927,"[서울대공원] 2025 장미원·식물원 축제 [장미, 정원을 품다]","테마가든 장미원, 식물원",NaN,37.4364305503019,127.014098361931
5014,[광진문화재단] 2025 나루 동요제,서울어린이대공원 열린무대,NaN,37.54895853939875,127.077746180044
5078,[광진문화재단] 2025 피크닉 in 나루,서울어린이대공원 숲속의무대,NaN,37.54885117753382,127.07998387392475
5223,[서울대공원] 2025 어린이날 기념행사 [모여라! 대공원],서울대공원 동물원 일대,NaN,37.4364305503019,127.014098361931
5603,[서울대공원] 2025 서울대공원 벚꽃축제,서울대공원 일대,NaN,37.4364305503019,127.014098361931


In [105]:
missing_gu["PLACE"].value_counts()

PLACE
서울동물원                                                               6
서울대공원 산림치유센터                                                        6
서울대공원                                                               5
서울대공원 치유숲                                                           5
서울대공원 서울동물원                                                         3
서울대공원 테마가든                                                          3
서울대공원 일대                                                            2
각 기관                                                                2
서울대공원 식물원, 식물표본전시관                                                  2
서울대공원 호숫가 둘레길                                                       2
서울대공원 테마가든 내 어린이동물원                                                 2
온라인                                                                 2
서울대공원 식물원                                                           2
주캐나다한국문화원 1층 전시실                                                    1
주워싱턴한국문화원 (237

### 동일 좌표의 정보로 결측값 대체

In [106]:
coord_to_gu = (
    df_c
    .dropna(subset=["GUNAME", "LAT", "LOT"])
    .drop_duplicates(subset=["LAT", "LOT"])
    .set_index(["LAT", "LOT"])["GUNAME"]
    .to_dict()
)

In [107]:
mask = df_c["GUNAME"].isna()

df_c.loc[mask, "GUNAME"] = (
    df_c.loc[mask]
    .apply(
        lambda x: coord_to_gu.get((x["LAT"], x["LOT"])),
        axis=1
    )
)

In [108]:
df_c["GUNAME"].isna().sum()

np.int64(3)

In [109]:
df_c[df_c["GUNAME"].isna()]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
197,전시/미술,NaN,"[서울시립미술관] 아마도, 모두 우리 [Perhaps, All of Us] 캐나다 전시",주캐나다한국문화원 1층 전시실,서울시립미술관,누구나,NaN,2026-08-10,기관,2026-08-25,2026-10-14,기타,75.6917°W,45.4215°N,무료,월 9:00~17:00 화-금 9:00~20:00 토 10:00~18:00 ※ 매주...,https://sema.seoul.go.kr/kr/whatson/exhibition...,https://culture.seoul.go.kr/culture/culture/cu...,2026
9342,축제-기타,NaN,2024 한강페스티벌 여름축제 [한강음악불꽃크루즈],아라김포여객터미널,한강사업본부,누구나,"대인 40,000원, 소인 25,000원",2024-07-02,기관,2024-07-27,2024-08-10,기타,126.786970046495,37.5976552679624,유료,매주 토요일 18:00~ (18:30 출항) (8.4.(일) 추가),https://hangang.seoul.go.kr/www/eventMng/detai...,https://culture.seoul.go.kr/culture/culture/cu...,2024
11725,클래식,NaN,서울시립교향악단 특별 음악회 얍 판 츠베덴의 바그너 [발퀴레],세종예술의전당,서울시립교향악단,8세이상 관람가능,"VIP석 100,000원, R석 80,000원, S석 60,000원",2024-01-09,시민,2024-02-02,2024-02-02,기타,127.267380381461,36.4866171885619,유료,19:30,https://www.seoulphil.or.kr/perf/view?perfNo=5...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [68]:
df_c.shape

(11934, 19)

In [110]:
df_c = df_c.dropna(subset='GUNAME')
df_c.shape

(11931, 19)

### 서울이 아닌 위치 확인 후 3개의 행사 제거

- 결측이 채워지지않는 좌표의 행사를 조회하여 위치를 파악 후 서울이 아닌 관계로 제거.

In [112]:
df_c[df_c['LOT'].isna()]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
4902,교육/체험,강서구,리드인 X 고명환 작가와 함께하는 [인문학 콘서트],강서시니어스타운 B1 송도아트홀,기타,독서논술 리드인 학부모님 또는 리드인 교육에 관심있는 분 누구나,무료,2025-04-24,시민,2025-06-03,2025-06-03,기타,NaN,NaN,무료,10:30~행사 종료시까지,https://blog.naver.com/readin6/223858472466,https://culture.seoul.go.kr/culture/culture/cu...,2025


In [113]:
df_c[df_c["PLACE"] == "강서시니어스타운 B1 송도아트홀"]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
4902,교육/체험,강서구,리드인 X 고명환 작가와 함께하는 [인문학 콘서트],강서시니어스타운 B1 송도아트홀,기타,독서논술 리드인 학부모님 또는 리드인 교육에 관심있는 분 누구나,무료,2025-04-24,시민,2025-06-03,2025-06-03,기타,NaN,NaN,무료,10:30~행사 종료시까지,https://blog.naver.com/readin6/223858472466,https://culture.seoul.go.kr/culture/culture/cu...,2025


In [114]:
df_c['LOT'] = df_c['LOT'].fillna(126.85943092649)
df_c['LAT'] = df_c['LAT'].fillna(37.557203266398)
df_c[df_c["PLACE"] == "강서시니어스타운 B1 송도아트홀"]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
4902,교육/체험,강서구,리드인 X 고명환 작가와 함께하는 [인문학 콘서트],강서시니어스타운 B1 송도아트홀,기타,독서논술 리드인 학부모님 또는 리드인 교육에 관심있는 분 누구나,무료,2025-04-24,시민,2025-06-03,2025-06-03,기타,126.859431,37.557203,무료,10:30~행사 종료시까지,https://blog.naver.com/readin6/223858472466,https://culture.seoul.go.kr/culture/culture/cu...,2025


### 강서구 좌표 결측을 장소를 조회하여 값을 대체

- 좌표가 결측인 값 1건을 검색하여 좌표 입력.

## 이상치 데이터 처리

In [117]:
df_c[df_c['PLACE'] == "갤러리 마리"]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
6224,전시/미술,종로구,푸른 뱀띠 해 특별전 [을사 1905-2025 : A New Dream in the...,갤러리 마리,기타,누구나,NaN,2025-02-14,시민,2025-01-17,2025-02-28,기타,-,-,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2025
7144,전시/미술,종로구,Beyond Genre [장르탈출 Ⅱ],갤러리 마리,기타,누구나,NaN,2024-12-14,시민,2024-11-22,2025-01-10,기타,-,-,무료,"화-토 11:00~19:00 / 일-월요일, 신정 1월 1일 휴무",http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
7956,전시/미술,종로구,"[추니박 개인전] 가보지 않은 길, 낯선 풍경",갤러리 마리,기타,누구나,NaN,2024-10-18,시민,2024-10-11,2024-11-15,기타,-,-,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
8757,전시/미술,종로구,"윤형선 [Dance of Flowers, Voice of Nature - 춤추는 꽃]",갤러리 마리,기타,누구나,NaN,2024-09-06,시민,2024-08-30,2024-10-04,기타,-,-,무료,"화~토 11:00~19:00 / 매주 일~월, 추석연휴 휴무",http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [118]:
df_c[df_c['LOT'] == "-"]

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
6224,전시/미술,종로구,푸른 뱀띠 해 특별전 [을사 1905-2025 : A New Dream in the...,갤러리 마리,기타,누구나,NaN,2025-02-14,시민,2025-01-17,2025-02-28,기타,-,-,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2025
7144,전시/미술,종로구,Beyond Genre [장르탈출 Ⅱ],갤러리 마리,기타,누구나,NaN,2024-12-14,시민,2024-11-22,2025-01-10,기타,-,-,무료,"화-토 11:00~19:00 / 일-월요일, 신정 1월 1일 휴무",http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
7956,전시/미술,종로구,"[추니박 개인전] 가보지 않은 길, 낯선 풍경",갤러리 마리,기타,누구나,NaN,2024-10-18,시민,2024-10-11,2024-11-15,기타,-,-,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
8757,전시/미술,종로구,"윤형선 [Dance of Flowers, Voice of Nature - 춤추는 꽃]",갤러리 마리,기타,누구나,NaN,2024-09-06,시민,2024-08-30,2024-10-04,기타,-,-,무료,"화~토 11:00~19:00 / 매주 일~월, 추석연휴 휴무",http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
9399,전시/미술,종로구,한불조형예술협회 [산책-PROMENADE],갤러리 마리,기타,누구나,NaN,2024-08-01,시민,2024-07-25,2024-08-16,기타,-,-,무료,화~토 11:00~19:00 / 매주 일~월 휴무,https://www.instagram.com/p/C9wlMzRS_bz/,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [119]:
lot_num = pd.to_numeric(df_c["LOT"], errors="coerce")
lat_num = pd.to_numeric(df_c["LAT"], errors="coerce")

invalid_lot = df_c[
    df_c["LOT"].notna() & lot_num.isna()
]

invalid_lat = df_c[
    df_c["LAT"].notna() & lat_num.isna()
]


In [120]:
invalid_lot["LOT"].value_counts()

LOT
-    5
Name: count, dtype: int64

In [121]:
invalid_lat["LAT"].value_counts()

LAT
-                     5
37.5718961547884~2    1
Name: count, dtype: int64

In [122]:
df_c[df_c['LAT'] == '37.5718961547884~2']

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
5195,축제-문화/예술,종로구,2025 국제기로 미술대축제,인사동 한국미술관 2층 전시관,기타,누구나,NaN,2025-04-21,시민,2025-05-07,2025-05-12,기타,126.987230558854,37.5718961547884~2,무료,10:00~18:00,https://www.kiroart.or.kr/,https://culture.seoul.go.kr/culture/culture/cu...,2025


In [ ]:
df_c["LAT"] = df_c["LAT"].replace("37.5718961547884~2", 37.57189615478842)
df_c["LAT"] = df_c["LAT"].replace("-", 37.5727115)
df_c["LOT"] = df_c["LOT"].replace("-", 126.9690368)


,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year


In [126]:
df_c[df_c["PLACE"] == '갤러리 마리']

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR,event_year
6224,전시/미술,종로구,푸른 뱀띠 해 특별전 [을사 1905-2025 : A New Dream in the...,갤러리 마리,기타,누구나,NaN,2025-02-14,시민,2025-01-17,2025-02-28,기타,126.969037,37.572711,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2025
7144,전시/미술,종로구,Beyond Genre [장르탈출 Ⅱ],갤러리 마리,기타,누구나,NaN,2024-12-14,시민,2024-11-22,2025-01-10,기타,126.969037,37.572711,무료,"화-토 11:00~19:00 / 일-월요일, 신정 1월 1일 휴무",http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
7956,전시/미술,종로구,"[추니박 개인전] 가보지 않은 길, 낯선 풍경",갤러리 마리,기타,누구나,NaN,2024-10-18,시민,2024-10-11,2024-11-15,기타,126.969037,37.572711,무료,화-토 11:00~19:00 / 일-월 휴무,http://www.gallerymarie.org/korean/viewtopic.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024
8757,전시/미술,종로구,"윤형선 [Dance of Flowers, Voice of Nature - 춤추는 꽃]",갤러리 마리,기타,누구나,NaN,2024-09-06,시민,2024-08-30,2024-10-04,기타,126.969037,37.572711,무료,"화~토 11:00~19:00 / 매주 일~월, 추석연휴 휴무",http://www.gallerymarie.org/korean/viewforum.p...,https://culture.seoul.go.kr/culture/culture/cu...,2024


In [127]:
df_c.to_csv("../data/cultural_events_2024_present.csv",
    index=False)

In [129]:
df_c["THEMECODE"].unique()

<StringArray>
['기타', '어린이/청소년 문화행사', '어르신 문화행사', '가족 문화행사', nan, '여성 문화행사', '문화가 있는 날']
Length: 7, dtype: str

# 인구 데이터 확인

In [ ]:
years = [2024, 2025, 2026]

population_dfs = {}

for year in years:
    
    # 1. CSV 불러오기
    df_1 = pd.read_csv(
        f"../data/등록인구(연령별_동별)_{year}.csv"
    )

    # 2. 0번 행을 컬럼명으로 지정
    df_1.columns = df_1.iloc[0]

    # 3. 기존 0번 행 제거
    df_1 = df_1.iloc[1:].reset_index(drop=True)

    # 4. 전체 합계 제외 + 한국인만 선택
    df_1 = df_1[
        (df_1["동별(1)"] != "합계") &
        (df_1["항목"] == "한국인")
    ].copy()

    # 5. 딕셔너리에 저장
    population_dfs[year] = df_1

In [137]:
rename_columns = {
    "동별(1)": "gu_name",
    "항목": "population_type",
    "합계": "total_population",
    "0~4세": "age_0_4",
    "5~9세": "age_5_9",
    "10~14세": "age_10_14",
    "15~19세": "age_15_19",
    "20~24세": "age_20_24",
    "25~29세": "age_25_29",
    "30~34세": "age_30_34",
    "35~39세": "age_35_39",
    "40~44세": "age_40_44",
    "45~49세": "age_45_49",
    "50~54세": "age_50_54",
    "55~59세": "age_55_59",
    "60~64세": "age_60_64",
    "65~69세": "age_65_69",
    "70~74세": "age_70_74",
    "75~79세": "age_75_79",
    "80~84세": "age_80_84",
    "85~89세": "age_85_89",
    "90~94세": "age_90_94",
    "95~99세": "age_95_99",
    "100세 이상": "age_100_plus",
    "연도": "year"
}

for year, df in population_dfs.items():
    df = df.rename(columns=rename_columns)
    df = df.drop(columns=["population_type"])
    
    population_dfs[year] = df

In [138]:
for year, df in population_dfs.items():
    df.to_csv(
        f"../data/population_{year}.csv",
        index=False,
        encoding="utf-8-sig"
    )